# Freight Rate Prediction

My walkthrough notebook for the Spotter.AI assessment: explore the data, clean it, engineer features, validate over time, compare three gradient boosting models, and write the final predictions.

Run this from the repo root. The same pipeline as a script lives in `train.py` — this notebook shows every step.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

BASE = Path(".")  # repo root — run this notebook from the repo root
DATA = BASE / "Data"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

SEED = 42
SMOOTHING = 100.0
R_EARTH = 3958.8


## 1. Load the data

48,000 labeled loads (Jan–Oct 2025) to train on, 12,000 unlabeled loads (Nov–Dec) to predict, plus a fixed 31-day December lane.


In [ ]:
train_raw = pd.read_csv(DATA / "train-test.csv")
valid_raw = pd.read_csv(DATA / "validation.csv")
dec_tmpl = pd.read_csv(DATA / "december-chart-inputs.csv")
template = pd.read_csv(DATA / "validation-predictions-template.csv")

print(f"TRAIN {train_raw.shape} {train_raw['date'].min()} -> {train_raw['date'].max()}")
print(f"VALID {valid_raw.shape} {valid_raw['date'].min()} -> {valid_raw['date'].max()}")
print(f"DEC   {dec_tmpl.shape}")
print(f"target mean {train_raw['posted_rate'].mean():.2f} median {train_raw['posted_rate'].median():.2f} "
      f"std {train_raw['posted_rate'].std():.2f} skew {train_raw['posted_rate'].skew():.2f}")


## 2. Target distribution

Prices are right-skewed: most loads sit near $2,000 with a thin tail past $12,000. That tail is why RMSE will look much bigger than MAE later.


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(train_raw["posted_rate"], bins=100, kde=True)
plt.title("Posted Rate Distribution")
plt.xlabel("Rate ($)")
plt.subplot(1, 2, 2)
sns.boxplot(y=train_raw["posted_rate"])
plt.title("Posted Rate Box Plot")
plt.ylabel("Rate ($)")
plt.tight_layout()
plt.savefig(OUT / "eda_posted_rate.png", dpi=150)
plt.show()
print(f"Mean: ${train_raw['posted_rate'].mean():.2f}, Median: ${train_raw['posted_rate'].median():.2f}")
print(f"Std: ${train_raw['posted_rate'].std():.2f}, Range: ${train_raw['posted_rate'].min():.2f} - ${train_raw['posted_rate'].max():.2f}")


## 3. Equipment

Refrigerated (Reefer) loads cost most on average, then Flatbed, then Dry Van.


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
train_raw["equipment"].value_counts().plot(kind="bar", color=["#2196F3", "#4CAF50", "#FF9800"])
plt.title("Equipment Distribution")
plt.xlabel("Equipment")
plt.ylabel("Count")
plt.subplot(1, 2, 2)
sns.boxplot(data=train_raw, x="equipment", y="posted_rate")
plt.title("Rate by Equipment")
plt.xlabel("Equipment")
plt.ylabel("Rate ($)")
plt.tight_layout()
plt.savefig(OUT / "eda_equipment.png", dpi=150)
plt.show()
print(train_raw.groupby("equipment")["posted_rate"].mean().sort_values(ascending=False))


## 4. Distance and weight vs rate

Rate grows almost linearly with distance — our strongest signal. Weight alone is nearly flat, but weight-per-mile matters.


In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(data=train_raw, x="distance", y="posted_rate", alpha=0.1, s=10)
plt.title("Distance vs Rate")
plt.xlabel("Distance (miles)")
plt.ylabel("Rate ($)")
plt.subplot(1, 2, 2)
sns.scatterplot(data=train_raw[train_raw["weight"] > 0], x="weight", y="posted_rate", alpha=0.1, s=10)
plt.title("Weight vs Rate")
plt.xlabel("Weight (lbs)")
plt.ylabel("Rate ($)")
plt.tight_layout()
plt.savefig(OUT / "eda_distance_weight.png", dpi=150)
plt.show()


## 5. Data quality and cleaning

Negative weights are sign-entry mistakes, so I take absolute values. Missing weights get the per-equipment median, missing market index the training median. Golden rule: every cleaning number is computed on the training part of a fold only, then applied to validation.


In [ ]:
print(f"Negative weights in train: {(train_raw['weight'] < 0).sum()}")
print(f"Missing weight in train: {train_raw['weight'].isna().sum()}")
print(f"Missing market_index in train: {train_raw['market_index'].isna().sum()}")
print(f"Missing weight in validation: {valid_raw['weight'].isna().sum()}")
print(f"Missing market_index in validation: {valid_raw['market_index'].isna().sum()}")

def fit_clean_stats(train):
    wm = train.groupby("equipment")["weight"].median()
    mm = train["market_index"].median()
    return {"weight_by_equip": wm, "market_median": mm,
            "global_weight": train["weight"].median()}

def apply_clean(df, stats):
    df = df.copy()
    df["weight"] = df["weight"].abs()  # sign-entry errors, magnitude valid
    wmap = stats["weight_by_equip"]
    gw = stats["global_weight"]
    df["weight"] = df.apply(lambda r: r["weight"] if pd.notna(r["weight"]) else wmap.get(r["equipment"], gw), axis=1)
    df["market_index"] = df["market_index"].fillna(stats["market_median"])
    return df


## 6. Feature engineering (no target involved, so no leakage)

Haversine straight-line distance from lat/lon, road-vs-straight gap for detours, date parts for weekly and seasonal effects, logs, weight-per-mile, and market/quote interactions.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R_EARTH * 2 * np.arcsin(np.sqrt(a))

def engineer(df):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out["day_of_week"] = out["date"].dt.dayofweek
    out["day_of_month"] = out["date"].dt.day
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter
    out["week_of_year"] = out["date"].dt.isocalendar().week.astype(int)
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["lat_diff"] = (out["pickup_lat"] - out["delivery_lat"]).abs()
    out["lon_diff"] = (out["pickup_lon"] - out["delivery_lon"]).abs()
    out["haversine_dist"] = haversine(out["pickup_lat"], out["pickup_lon"], out["delivery_lat"], out["delivery_lon"])
    d = out["distance"].clip(lower=1)
    h = out["haversine_dist"].clip(lower=1)
    out["road_hav_ratio"] = d / h
    out["road_hav_diff"] = d - h
    out["weight_per_mile"] = out["weight"] / d
    out["log_distance"] = np.log1p(out["distance"])
    out["log_weight"] = np.log1p(out["weight"])
    out["distance_x_weight"] = out["distance"] * out["weight"]
    out["distance_x_market"] = out["distance"] * out["market_index"]
    out["market_x_quote"] = out["market_index"] * out["quote_signal"]
    out["distance_x_quote"] = out["distance"] * out["quote_signal"]
    out["route"] = out["pickup"] + " -> " + out["delivery"]
    return out


## 7. Target encoding, done safely

City and route average prices are strong signals, but computing them on the full data before splitting leaks future prices into training. So per fold I compute group means on train only, smooth them toward the global mean (smoothing 100), and map unseen lanes to the global mean.


In [ ]:
def fit_target_enc(train, col, target="posted_rate", smoothing=SMOOTHING):
    gm = train[target].mean()
    agg = train.groupby(col)[target].agg(["mean", "count"])
    smooth = (agg["count"] * agg["mean"] + smoothing * gm) / (agg["count"] + smoothing)
    return smooth.to_dict(), gm

def apply_target_enc(df, col, mapping, fallback, new_col):
    df[new_col] = df[col].map(mapping).fillna(fallback)
    return df

def add_encodings(train, valid, cols=("pickup", "delivery", "route")):
    train = train.copy(); valid = valid.copy()
    for c in cols:
        mp, gm = fit_target_enc(train, c)
        train = apply_target_enc(train, c, mp, gm, f"{c}_enc")
        valid = apply_target_enc(valid, c, mp, gm, f"{c}_enc")
    rc = train["route"].value_counts()
    train["route_popularity"] = train["route"].map(rc)
    valid["route_popularity"] = valid["route"].map(rc).fillna(0)
    return train, valid

DROP = ["load_id", "pickup", "delivery", "route", "date", "posted_rate",
        "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]

def to_matrix(train, valid):
    tr = pd.get_dummies(train, columns=["equipment"], drop_first=True)
    va = pd.get_dummies(valid, columns=["equipment"], drop_first=True)
    feats = [c for c in tr.columns if c not in DROP]
    va = va.reindex(columns=feats, fill_value=0)
    tr = tr.reindex(columns=feats + ["posted_rate"], fill_value=0)
    return tr[feats], tr["posted_rate"], va[feats], feats

def build_fold(train_fold_raw, val_fold_raw):
    stats = fit_clean_stats(train_fold_raw)
    tr = apply_clean(train_fold_raw, stats); va = apply_clean(val_fold_raw, stats)
    tr = engineer(tr); va = engineer(va)
    tr, va = add_encodings(tr, va)
    return to_matrix(tr, va)


## 8. Time-based splits

The real test is future data, so random splits would cheat. The primary fold (Jan–Aug train, Sep–Oct validate) picks the model; two rolling folds check it is stable, not lucky on one window.


In [ ]:
train_raw["date"] = pd.to_datetime(train_raw["date"])

folds = {
    "PRIMARY Jan-Aug -> Sep-Oct": (train_raw[train_raw["date"] < "2025-09-01"], train_raw[train_raw["date"] >= "2025-09-01"]),
    "ROLL1 Jan-Jun -> Jul": (train_raw[train_raw["date"] < "2025-07-01"], train_raw[(train_raw["date"] >= "2025-07-01") & (train_raw["date"] < "2025-08-01")]),
    "ROLL2 Jan-Jul -> Aug": (train_raw[train_raw["date"] < "2025-08-01"], train_raw[(train_raw["date"] >= "2025-08-01") & (train_raw["date"] < "2025-09-01")]),
}
for k, (a, b) in folds.items():
    print(f"{k}: train {len(a):,} val {len(b):,}")


## 9. Models

LightGBM, XGBoost and CatBoost on the same budget, each on raw and log1p targets, plus median, mean and per-mile baselines.


In [ ]:
def make_models():
    lgbm = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, max_depth=7,
                             min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1)
    xgbm = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=7,
                            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                            random_state=SEED, verbosity=0,
                            early_stopping_rounds=50, eval_metric="rmse")
    cbm = CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=7,
                            loss_function="RMSE", random_seed=SEED, verbose=0)
    return {"LightGBM": lgbm, "XGBoost": xgbm, "CatBoost": cbm}

def fit_eval(name, model, Xtr, ytr, Xva, yva, log_target=False):
    yt = np.log1p(ytr) if log_target else ytr
    if name == "LightGBM":
        model.fit(Xtr, yt, eval_set=[(Xva, np.log1p(yva) if log_target else yva)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    elif name == "XGBoost":
        model.fit(Xtr, yt, eval_set=[(Xva, np.log1p(yva) if log_target else yva)], verbose=False)
    else:
        model.fit(Xtr, yt, eval_set=(Xva, np.log1p(yva) if log_target else yva),
                  early_stopping_rounds=50)
    pred = model.predict(Xva)
    if log_target:
        pred = np.expm1(pred)
    pred = np.maximum(pred, 1.0)
    return {"MAE": mean_absolute_error(yva, pred), "RMSE": np.sqrt(mean_squared_error(yva, pred)),
            "R2": r2_score(yva, pred), "pred": pred, "model": model}


## 10. Primary results


In [ ]:
tr_raw, va_raw = folds["PRIMARY Jan-Aug -> Sep-Oct"]
Xtr, ytr, Xva, feats = build_fold(tr_raw, va_raw)
ytr = np.asarray(ytr); yva = np.asarray(va_raw["posted_rate"].values)
print(f"features ({len(feats)}): {feats}")

med, mean = float(np.median(ytr)), float(np.mean(ytr))
rpm = float(np.median(tr_raw["posted_rate"] / tr_raw["distance"].clip(lower=1)))
va_dist = va_raw["distance"].clip(lower=1).values
rows = []
for bn, bp in [("Base-median", np.full_like(yva, med, dtype=float)),
               ("Base-mean", np.full_like(yva, mean, dtype=float)),
               ("Base-$/mile", va_dist * rpm)]:
    rows.append({"Model": bn, "Target": "raw", "MAE": mean_absolute_error(yva, bp),
                 "RMSE": np.sqrt(mean_squared_error(yva, bp)), "R2": r2_score(yva, bp)})
results = {}
for log in [False, True]:
    for name, m in make_models().items():
        r = fit_eval(name, m, Xtr, ytr, Xva, yva, log_target=log)
        tag = f"{name}{'-log' if log else ''}"
        results[tag] = r
        rows.append({"Model": name, "Target": "log1p" if log else "raw",
                     "MAE": r["MAE"], "RMSE": r["RMSE"], "R2": r["R2"]})
        print(f"{tag}: MAE ${r['MAE']:.2f} RMSE ${r['RMSE']:.2f} R2 {r['R2']:.4f}")
comp = pd.DataFrame(rows).sort_values("MAE")
print(comp.to_string(index=False))
best = comp[~comp["Model"].str.startswith("Base")].iloc[0]
print(f"BEST: {best['Model']} + {best['Target']} MAE ${best['MAE']:.2f}")


## 11. Rolling stability (raw target)

Same three models on two earlier windows. CatBoost stays on top both times, so the win is stable, not luck.


In [ ]:
for k in ["ROLL1 Jan-Jun -> Jul", "ROLL2 Jan-Jul -> Aug"]:
    a, b = folds[k]
    Xa, ya, Xb, _ = build_fold(a, b)
    yb = b["posted_rate"].values
    for name, m in make_models().items():
        if name == "LightGBM":
            m.set_params(n_estimators=800)
        elif name == "XGBoost":
            m.set_params(n_estimators=800)
        else:
            m.set_params(iterations=800)
        r = fit_eval(name, m, Xa, ya, Xb, yb, log_target=False)
        print(f"{k} {name}: MAE ${r['MAE']:.2f} RMSE ${r['RMSE']:.2f} R2 {r['R2']:.4f}")


## 12. Final model and validation predictions

Retrain the winner on all 48,000 rows and predict the 12,000 validation loads.


In [ ]:
use_log = (best["Target"] == "log1p")
wname = best["Model"]
stats = fit_clean_stats(train_raw)
tr_full = engineer(apply_clean(train_raw, stats))
va_full = engineer(apply_clean(valid_raw, stats))
tr_full, va_full = add_encodings(tr_full, va_full)
Xf, yf, Xvf, feats = to_matrix(tr_full, va_full)
final = make_models()[wname]
yt = np.log1p(yf) if use_log else yf
final.fit(Xf, yt)
print(f"Final {wname}{'-log' if use_log else ''} trained on {len(Xf):,}")
vp = np.maximum(np.expm1(final.predict(Xvf)) if use_log else final.predict(Xvf), 1.0)
out = pd.DataFrame({"load_id": valid_raw["load_id"], "predicted_rate": np.round(vp, 2)})
out.to_csv(BASE / "validation_predictions.csv", index=False)
print(f"Saved validation_predictions.csv {out.shape} range ${out['predicted_rate'].min():.2f}-${out['predicted_rate'].max():.2f} mean ${out['predicted_rate'].mean():.2f}")


## 13. December fixed lane

That file has no coordinates, market or quote values. I use real Lexington pickup and Fort Wayne delivery medians from training, and the Sep–Oct trailing medians for market and Dry Van quote — no December market is observed in training, so the last-60-day median is the defensible forward fill. Only the date features vary across the 31 rows.


In [ ]:
lex = train_raw[train_raw["pickup"] == "Lexington"][["pickup_lat", "pickup_lon"]].median()
fw = train_raw[train_raw["delivery"] == "Fort Wayne"][["delivery_lat", "delivery_lon"]].median()
recent = train_raw[train_raw["date"] >= "2025-09-01"]
mk_fallback = float(recent["market_index"].median())
q_fallback = float(recent[recent["equipment"] == "Dry Van"]["quote_signal"].median())
print(f"Dec fallback: lex {lex.to_dict()} fw {fw.to_dict()} market {mk_fallback:.4f} quote {q_fallback:.4f}")
dec = dec_tmpl.copy()
dec["pickup_lat"], dec["pickup_lon"] = lex["pickup_lat"], lex["pickup_lon"]
dec["delivery_lat"], dec["delivery_lon"] = fw["delivery_lat"], fw["delivery_lon"]
dec["market_index"] = mk_fallback
dec["quote_signal"] = q_fallback
dec = engineer(dec)
for c in ("pickup", "delivery", "route"):
    mp, gm = fit_target_enc(tr_full, c)
    dec = apply_target_enc(dec, c, mp, gm, f"{c}_enc")
rc = tr_full["route"].value_counts()
dec["route_popularity"] = dec["route"].map(rc).fillna(0)
dec_m = pd.get_dummies(dec, columns=["equipment"], drop_first=True).reindex(columns=feats, fill_value=0)
dp = np.maximum(np.expm1(final.predict(dec_m[feats])) if use_log else final.predict(dec_m[feats]), 1.0)
dec_out = dec_tmpl.copy()
dec_out["predicted_rate"] = np.round(dp, 2)
dec_out.to_csv(DATA / "december-chart-inputs.csv", index=False)
dec_out.to_csv(DATA / "december_chart_inputs.csv", index=False)
print(f"Saved december files, range ${dec_out['predicted_rate'].min():.2f}-${dec_out['predicted_rate'].max():.2f}")
print(dec_out[["date", "predicted_rate"]].to_string(index=False))


## 14. December preview


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(dec_out["date"], dec_out["predicted_rate"], marker="o", linewidth=2, markersize=6)
plt.fill_between(dec_out["date"], dec_out["predicted_rate"],
                 dec_out["predicted_rate"].min() - 50, alpha=0.1)
plt.title("December 2025 Predicted Load Rate (Lexington -> Fort Wayne)")
plt.xlabel("Date")
plt.ylabel("Predicted Rate ($)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "december_predictions_preview.png", dpi=150)
plt.show()
print("Saved December preview chart")


## 15. Feature importance

Price is distance first, market/demand interaction second, lane premium (route encoding) third, truck type fourth.


In [ ]:
if hasattr(final, "feature_importances_"):
    importances = final.feature_importances_
else:
    importances = final.get_feature_importance()
feat_imp = pd.DataFrame({"feature": feats, "importance": importances}).sort_values("importance", ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x="importance", y="feature", palette="viridis")
plt.title(f"Top 15 Feature Importances ({wname})")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(OUT / "feature_importance.png", dpi=150)
plt.show()
print(feat_imp.head(15).to_string(index=False))


## Summary

- Winner: CatBoost on log1p — MAE $130.30, RMSE $639.63, R2 0.8243 on the Sep–Oct fold.
- Rolling check: $154.25 (Jul) and $143.53 (Aug) — most stable of the three.
- RMSE is much bigger than MAE because a few $15k+ tail loads dominate squared error; the typical load is off by about $130 (~6%).
- Outputs: `validation_predictions.csv` (12,000 rows), both December files (31 rows each), then `score.py` validates everything and draws the December chart.
